# Transform Step

This notebook transforms the raw sales data by:

1. **Deduplication**: Remove duplicate order records
2. **Data cleaning**: Trim whitespace, standardize text columns
3. **Derived columns**: Calculate `total_amount` = quantity × unit_price
4. **Enrichment**: Add `order_month` and `order_year` for time-based analysis
5. **Data quality**: Filter out invalid records (null order_id, negative quantities)

**Input**: `raw_sales` temp view from the Extract step
**Output**: `transformed_sales_df` DataFrame for the Load step

In [0]:
# Databricks notebook source
# --- Transform: Clean, enrich, and validate sales data ---

from pyspark.sql.functions import col, trim, upper, month, year, date_format, when, sum as _sum

# Read the raw data from the temp view created in the Extract step
raw_sales_df = spark.table("raw_sales")

# 1. Remove exact duplicate rows
deduped_df = raw_sales_df.distinct()
print(f"After deduplication: {deduped_df.count()} rows (was {raw_sales_df.count()})")

# 2. Clean text columns: trim whitespace and standardize casing
cleaned_df = (
    deduped_df
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("region", upper(trim(col("region"))))
)

# 3. Data quality: filter out invalid records
valid_df = cleaned_df.filter(
    col("order_id").isNotNull()
    & (col("quantity") > 0)
    & (col("unit_price") > 0)
)
removed_count = cleaned_df.count() - valid_df.count()
print(f"Removed {removed_count} invalid records")

# 4. Add derived columns: total_amount, order_month, order_year
transformed_sales_df = (
    valid_df
    .withColumn("total_amount", (col("quantity") * col("unit_price")).cast("decimal(10,2)"))
    .withColumn("order_month", month(col("order_date")))
    .withColumn("order_year", year(col("order_date")))
    .withColumn("order_date_str", date_format(col("order_date"), "yyyy-MM-dd"))
)

# 5. Print summary statistics
print(f"\nTransformed row count: {transformed_sales_df.count()}")
print("\nSales by region:")
transformed_sales_df.groupBy("region").agg(
    _sum("total_amount").alias("total_revenue"),
    col("region")
).orderBy(col("total_revenue").desc()).show()

print("\nSales by category:")
transformed_sales_df.groupBy("category").agg(
    _sum("total_amount").alias("total_revenue")
).orderBy(col("total_revenue").desc()).show()

print("\nSample transformed data:")
transformed_sales_df.show(5, truncate=False)

# Make available to downstream notebooks via temp view
transformed_sales_df.createOrReplaceTempView("transformed_sales")

print("Transform step completed successfully.")